# Create a CORPUS table from the novels

## Get the data

In [14]:
import pandas as pd
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

directory_path  = 'C:/Users/mason/Box/Text As Data Final/Output' 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mason\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mason\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mason\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mason\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [17]:
LIB = pd.read_csv(f"{directory_path}/BrandonSanderson_LIB.csv")
CLEAN_TEXT = pd.read_csv(f"{directory_path}/BrandonSanderson_CLEAN_TEXT.csv")

CLEAN_TEXT['chapter_id'] = CLEAN_TEXT.groupby('title')['chapter_id'].transform(lambda x: pd.factorize(x)[0])

CLEAN_TEXT['paragraph_id'] = CLEAN_TEXT.groupby(['title', 'chapter_id']).cumcount()

## Break down the paragraphs into sentences and terms

The Dataframe (`books_df`) as it currently stands utilized the html like parsing of `'p'` to denote paragraphs so rather than throw those away they were added to the dataframe. One less OHCO step down the line.

Columns as delimitted column names, including `token_str`, `term_str`, `pos`, and `pos_group`

In [24]:

OHCO_SENTS=['title', 'chapter_id', 'paragraph_id', 'sent_id']
OHCO = ['title', 'chapter_id', 'paragraph_id', 'sent_id', 'token_id'] 

CLEAN_TEXT['sentence'] = CLEAN_TEXT['text'].apply(nltk.sent_tokenize)
df_sentences = CLEAN_TEXT.explode('sentence').drop(columns=['text'])

df_sentences = df_sentences[df_sentences['sentence'].str.strip() != '']

df_sentences['sent_id'] = df_sentences.groupby(['title', 'chapter_id', 'paragraph_id']).cumcount()

df_sentences = df_sentences.set_index(OHCO_SENTS)

TOKENS = df_sentences['sentence'].apply(
    lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x)))
    ).stack().dropna().to_frame('pos_tuple')

TOKENS.index.names = OHCO

TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS = TOKENS.drop('pos_tuple', axis=1)

TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex=True)

CORPUS = TOKENS[TOKENS.term_str != ''].copy()

CORPUS = CORPUS.reset_index()

CORPUS = CORPUS[['title', 'chapter_id', 'paragraph_id', 'sent_id', 'token_id', 'token_str', 'term_str', 'pos', 'pos_group']]

with pd.option_context('display.max_rows', 20):
    print(CORPUS.head(20))

                title  chapter_id  paragraph_id  sent_id  token_id token_str  \
0   A Memory of Light           0             0        0         0       The   
1   A Memory of Light           0             0        0         1     Wheel   
2   A Memory of Light           0             0        0         2        of   
3   A Memory of Light           0             0        0         3      Time   
4   A Memory of Light           0             0        0         4     turns   
5   A Memory of Light           0             0        0         6       and   
6   A Memory of Light           0             0        0         7      Ages   
7   A Memory of Light           0             0        0         8      come   
8   A Memory of Light           0             0        0         9       and   
9   A Memory of Light           0             0        0        10      pass   
10  A Memory of Light           0             0        0        12   leaving   
11  A Memory of Light           0       

## Save the CORPUS

In [25]:
CORPUS.to_csv(f"{directory_path}/BrandonSanderson_CORPUS.csv")
print("Tokens saved to CSV!")

Tokens saved to CSV!


In [26]:
len(CORPUS)


2224830

In [29]:
#get the column names of the corpus
CORPUS.columns.tolist()

['title',
 'chapter_id',
 'paragraph_id',
 'sent_id',
 'token_id',
 'token_str',
 'term_str',
 'pos',
 'pos_group']